# 15.9 Graphs

**Prerequisites:** 15.5 Stacks and Queues, 15.7 Trees, 15.8 Heaps  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Adjacency **list** vs **matrix**, and when each wins
- **BFS** with a queue, **DFS** with a stack - the same code, one structure apart
- 🔴 The **visited set**: without it, a cycle loops forever
- Shortest path in an unweighted graph, free from BFS
- **Dijkstra** with a heap (**15.8**) - and why it fails on negative weights
- **Topological sort** for dependency order
- Cycle detection - and why directed and undirected need different algorithms
- Connected components, and the union-find preview
- Interview questions, worked

---

## A graph is the general case

A linked list (**15.4**) is a node with one successor. A tree (**15.7**) has several, with no cycles and one root. A **graph** drops both restrictions: any node may connect to any other, cycles are allowed, and there need be no root.

```
   LIST      A → B → C

   TREE          A            one root, no cycles
                / \
               B   C

   GRAPH     A ──→ B          cycles allowed
             ↑     │          no root
             └── C ←┘         any node may reach any other
```

| Word | Means |
|---|---|
| **vertex** / node | a thing |
| **edge** | a connection between two things |
| **directed** | edges have a direction (`A → B` ≠ `B → A`) |
| **weighted** | edges carry a cost — distance, latency, price |
| **degree** | how many edges touch a vertex (in-degree / out-degree if directed) |
| **path** | a sequence of edges from one vertex to another |
| **cycle** | a path returning to where it started |
| **connected** | every vertex reachable from every other |
| **DAG** | directed acyclic graph — directed, no cycles |

> **You have already met graphs twice.** Folder **10.6** modelled a service dependency graph and traversed it with recursive SQL and Cypher; folder **11** was about a network, which is a graph of machines. This notebook is the algorithms underneath both.

## Representation decides everything

### Adjacency list — a dict of neighbours

```
   {"A": ["B", "C"],
    "B": ["D"],
    "C": ["D"],
    "D": []}
```

### Adjacency matrix — a grid of booleans

```
        A  B  C  D
     A  0  1  1  0
     B  0  0  0  1
     C  0  0  0  1
     D  0  0  0  0
```

| | Adjacency list | Adjacency matrix |
|---|---|---|
| Space | **O(V + E)** | O(V²) |
| "Is there an edge A→B?" | O(degree) | **O(1)** |
| "List A's neighbours" | **O(degree)** | O(V) |
| Add an edge | O(1) | O(1) |
| Best for | **sparse** graphs | **dense** graphs |

🔴 **Real graphs are almost always sparse.** A social network with a billion users has perhaps a few hundred edges each — a matrix would need 10¹⁸ cells to store 10¹¹ edges. **Default to the adjacency list**; reach for a matrix only when the graph is dense, tiny, or you need O(1) edge lookups.

In Python, `defaultdict(list)` (**15.6**) is the natural adjacency list.

In [ ]:
from collections import defaultdict, deque


class Graph:
    """Adjacency-list graph. Directed by default."""

    def __init__(self, directed=True):
        self.adjacent = defaultdict(list)
        self.directed = directed
        self.vertices = set()

    def add_edge(self, source, target, weight=1):
        self.adjacent[source].append((target, weight))
        self.vertices.update((source, target))
        if not self.directed:
            self.adjacent[target].append((source, weight))

    def neighbours(self, vertex):
        return [target for target, _weight in self.adjacent[vertex]]

    def weighted_neighbours(self, vertex):
        return self.adjacent[vertex]

    def __repr__(self):
        kind = "directed" if self.directed else "undirected"
        edges = sum(len(v) for v in self.adjacent.values())
        if not self.directed:
            edges //= 2
        return f"<Graph {kind}: {len(self.vertices)} vertices, {edges} edges>"


# The service dependency graph from 10.6, so the two notebooks line up.
EDGES = [
    ("web", "api"), ("web", "cdn"),
    ("api", "auth"), ("api", "cache"), ("api", "search"),
    ("auth", "userdb"), ("cache", "userdb"),
    ("search", "index"), ("index", "userdb"),
]

services = Graph(directed=True)
for source, target in EDGES:
    services.add_edge(source, target)

print(services)
for vertex in sorted(services.vertices):
    print(f"  {vertex:<8} -> {services.neighbours(vertex)}")

# the matrix form, for contrast
order = sorted(services.vertices)
index = {name: i for i, name in enumerate(order)}
matrix = [[0] * len(order) for _ in order]
for source, target in EDGES:
    matrix[index[source]][index[target]] = 1

print(f"\nas a matrix ({len(order)}x{len(order)} = {len(order) ** 2} cells "
      f"for {len(EDGES)} edges):")
print("        " + " ".join(f"{name[:3]:>4}" for name in order))
for name in order:
    row = " ".join(f"{cell:>4}" for cell in matrix[index[name]])
    print(f"  {name:<7}{row}")
print("\n  Mostly zeros - which is what 'sparse' means, and why the list wins.")

---

# Traversal: BFS and DFS

Two ways to visit everything reachable. **The code is nearly identical** — they differ only in which container holds the frontier.

```
   BFS                                DFS
   frontier = deque([start])          frontier = [start]
   node = frontier.popleft()   <-->   node = frontier.pop()
        ^^^^^^^^ a QUEUE                    ^^^ a STACK
```

That one line changes the whole character:

| | **BFS** | **DFS** |
|---|---|---|
| Structure | queue (**15.5**) | stack, or recursion |
| Explores | level by level, nearest first | one path to the end, then backtracks |
| Memory | O(width) — can be huge | O(depth) |
| **Shortest path (unweighted)** | ✅ **yes, guaranteed** | 🔴 no |
| Good for | shortest path, nearest neighbours, levels | cycle detection, topological sort, path existence |

> **Why BFS gives shortest paths and DFS does not:** BFS reaches everything at distance 1 before anything at distance 2. The first time it sees a vertex is therefore by a shortest route. DFS may plunge down a long path and arrive at the same vertex the long way round.

### 🔴 The visited set is not optional

A tree has no cycles, so a traversal terminates naturally. **A graph does not.** Without a visited set, `A → B → C → A` loops forever.

```
    A → B → C
    ↑       │
    └───────┘        visit A, B, C, A, B, C, ... forever
```

🔴 And **mark as visited when you enqueue, not when you dequeue**. Otherwise a vertex with several predecessors is added to the queue several times before any of those copies is processed — the traversal still terminates, but does redundant work and can blow up on dense graphs.

The cell below runs an unguarded traversal with a hard step limit, to show it never stops on its own.

In [ ]:
cyclic = Graph(directed=True)
for source, target in (("A", "B"), ("B", "C"), ("C", "A")):
    cyclic.add_edge(source, target)


def traverse_unguarded(graph, start, limit=20):
    """No visited set. The limit is the only reason this returns."""
    order, frontier = [], deque([start])
    while frontier and len(order) < limit:
        node = frontier.popleft()
        order.append(node)
        frontier.extend(graph.neighbours(node))
    return order, bool(frontier)


order, still_going = traverse_unguarded(cyclic, "A")
print("without a visited set:")
print("  visited:", " ".join(order))
print(f"  stopped only because of the step limit: {still_going}")
print("  🔴 In real code with no limit, this never returns.\n")


def bfs(graph, start):
    """Breadth-first. Mark visited when ENQUEUEING, not when dequeueing."""
    visited = {start}
    order, frontier = [], deque([start])
    while frontier:
        node = frontier.popleft()          # QUEUE -> breadth-first
        order.append(node)
        for neighbour in graph.neighbours(node):
            if neighbour not in visited:
                visited.add(neighbour)     # 🔴 here, not after popleft
                frontier.append(neighbour)
    return order


def dfs_iterative(graph, start):
    """Depth-first. The SAME code with a stack instead of a queue."""
    visited = set()
    order, frontier = [], [start]
    while frontier:
        node = frontier.pop()              # STACK -> depth-first
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for neighbour in reversed(graph.neighbours(node)):
            if neighbour not in visited:
                frontier.append(neighbour)
    return order


def dfs_recursive(graph, node, visited=None, order=None):
    visited = set() if visited is None else visited
    order = [] if order is None else order
    visited.add(node)
    order.append(node)
    for neighbour in graph.neighbours(node):
        if neighbour not in visited:
            dfs_recursive(graph, neighbour, visited, order)
    return order


print("with a visited set, on the cyclic graph:")
print("  bfs:", bfs(cyclic, "A"), "- terminates\n")

print("on the service graph:")
print("  bfs          :", bfs(services, "web"))
print("  dfs iterative:", dfs_iterative(services, "web"))
print("  dfs recursive:", dfs_recursive(services, "web"))
print("\n  BFS visits cdn early (distance 1). DFS dives down api first.")

## Shortest path in an unweighted graph

BFS gives it for free. Record each vertex's **parent** as you discover it, then walk the parents backwards from the destination.

```
    parent = {"web": None, "api": "web", "auth": "api", "userdb": "auth"}

    userdb -> auth -> api -> web        then reverse
```

This is the **same answer** the recursive CTE and Cypher `shortestPath` produced in **10.6** — and, as noted there, a shortest path is **not unique**. Any two of equal length are equally correct, so test on length, never identity.

| | Time | Space |
|---|---|---|
| BFS shortest path | O(V + E) | O(V) |

In [ ]:
def shortest_path(graph, start, goal):
    """BFS shortest path in an UNWEIGHTED graph. O(V + E)."""
    if start == goal:
        return [start]
    parent = {start: None}
    frontier = deque([start])
    while frontier:
        node = frontier.popleft()
        for neighbour in graph.neighbours(node):
            if neighbour in parent:
                continue
            parent[neighbour] = node
            if neighbour == goal:
                path = [goal]
                while parent[path[-1]] is not None:
                    path.append(parent[path[-1]])
                return path[::-1]
            frontier.append(neighbour)
    return None


def distances_from(graph, start):
    """Hops to every reachable vertex - BFS level by level (15.7)."""
    distance = {start: 0}
    frontier = deque([start])
    while frontier:
        node = frontier.popleft()
        for neighbour in graph.neighbours(node):
            if neighbour not in distance:
                distance[neighbour] = distance[node] + 1
                frontier.append(neighbour)
    return distance


path = shortest_path(services, "web", "userdb")
print("shortest web -> userdb:", " -> ".join(path))
print(f"  {len(path) - 1} hops")
print("  🔴 Not unique: web->api->cache->userdb is equally short (10.6).")

print("\nhops from 'web' to everything:")
for vertex, hops in sorted(distances_from(services, "web").items(),
                          key=lambda pair: (pair[1], pair[0])):
    print(f"  {hops} hop(s): {vertex}")

print("\nunreachable:", shortest_path(services, "userdb", "web"))
print("  ^ the graph is DIRECTED - userdb has no outgoing edges.")

## Dijkstra - shortest path with weights

Once edges have **costs**, BFS is wrong: the fewest hops may not be the cheapest route.

```
        A ──1──> B ──1──> D          A->B->D costs 2 over 2 hops
        │                 ▲
        └────────9────────┘          A->D costs 9 over 1 hop

   BFS picks A->D (1 hop). Dijkstra picks A->B->D (cost 2).
```

**Dijkstra** is BFS with a **priority queue** (**15.8**) instead of a plain queue: always expand the cheapest-known frontier vertex next.

| | Complexity |
|---|---|
| With a binary heap | **O((V + E) log V)** |

🔴 **Dijkstra requires non-negative weights.** With a negative edge, a vertex already finalised might later be reachable more cheaply — and Dijkstra never revisits it. Use **Bellman-Ford** (O(V·E)) when negatives are possible; it also detects negative cycles.

The implementation below uses **lazy deletion** (**15.8**): rather than decreasing a key in the heap, push a new entry and skip stale ones on the way out.

In [ ]:
import heapq


def dijkstra(graph, start):
    """Cheapest cost to every reachable vertex. Non-negative weights only."""
    best = {start: 0}
    parent = {start: None}
    finalised = set()
    heap = [(0, start)]
    while heap:
        cost, node = heapq.heappop(heap)
        if node in finalised:
            continue                       # a stale entry - lazy deletion (15.8)
        finalised.add(node)
        for neighbour, weight in graph.weighted_neighbours(node):
            candidate = cost + weight
            if candidate < best.get(neighbour, float("inf")):
                best[neighbour] = candidate
                parent[neighbour] = node
                heapq.heappush(heap, (candidate, neighbour))
    return best, parent


def rebuild(parent, goal):
    if goal not in parent:
        return None
    path = [goal]
    while parent[path[-1]] is not None:
        path.append(parent[path[-1]])
    return path[::-1]


network = Graph(directed=True)
for source, target, weight in [
    ("A", "B", 1), ("B", "D", 1), ("A", "D", 9),
    ("A", "C", 2), ("C", "D", 4), ("D", "E", 3), ("C", "E", 8),
]:
    network.add_edge(source, target, weight)

costs, parents = dijkstra(network, "A")
print("cheapest cost from A:")
for vertex in sorted(costs):
    print(f"  {vertex}: cost {costs[vertex]}   via {' -> '.join(rebuild(parents, vertex))}")

hops = shortest_path(network, "A", "D")
print(f"\nBFS (fewest hops)  : {' -> '.join(hops)}  = {len(hops) - 1} hop(s), "
      f"cost 9")
print(f"Dijkstra (cheapest): {' -> '.join(rebuild(parents, 'D'))}  = cost "
      f"{costs['D']}")
print("\n🔴 Fewest hops is not cheapest. Which you want depends entirely on")
print("   what the weights mean - latency, price, distance.")

In [ ]:
# 🔴 Why Dijkstra breaks on negative weights - demonstrated properly.
#
# The damage needs a vertex AFTER the one that improves. Here B is
# finalised at cost 2, D is computed from it as 3, and when C later shows
# B really costs 1, D is never recomputed - B is already finalised.
negative = Graph(directed=True)
for source, target, weight in [
    ("A", "B", 2),
    ("A", "C", 5),
    ("C", "B", -4),      # makes B cheaper, but only after B is finalised
    ("B", "D", 1),       # D inherits the stale cost
]:
    negative.add_edge(source, target, weight)

costs, _ = dijkstra(negative, "A")
true_cost_to_d = 5 + (-4) + 1                 # A -> C -> B -> D

print("graph: A->B (2), A->C (5), C->B (-4), B->D (1)\n")
print(f"  dijkstra says D costs : {costs['D']}")
print(f"  A -> C -> B -> D costs: {true_cost_to_d}")
print(f"  wrong by              : {costs['D'] - true_cost_to_d}")
print()
print("🔴 Dijkstra finalised B at 2 and computed D = 3 from it. C then")
print("   showed B really costs 1 - but B was finalised, so D was never")
print("   recomputed. The algorithm ASSUMES costs only ever go up as you")
print("   move outward, and a negative edge breaks that assumption.\n")


def bellman_ford(graph, start):
    """Handles negative weights, and detects negative cycles. O(V*E)."""
    best = {vertex: float("inf") for vertex in graph.vertices}
    best[start] = 0
    edges = [(source, target, weight)
             for source in graph.adjacent
             for target, weight in graph.adjacent[source]]

    for _ in range(len(graph.vertices) - 1):        # V-1 relaxation rounds
        changed = False
        for source, target, weight in edges:
            if best[source] + weight < best[target]:
                best[target] = best[source] + weight
                changed = True
        if not changed:
            break                                   # settled early

    for source, target, weight in edges:            # one more round finds a cycle
        if best[source] + weight < best[target]:
            raise ValueError("negative cycle detected")
    return best


settled = bellman_ford(negative, "A")
print("bellman_ford says:", {k: settled[k] for k in sorted(settled)})
print(f"  D costs {settled['D']} - correct, because it relaxes EVERY edge")
print("  repeatedly instead of finalising vertices one at a time.\n")

loop = Graph(directed=True)
for source, target, weight in [("A", "B", 1), ("B", "C", -2), ("C", "A", -1)]:
    loop.add_edge(source, target, weight)
try:
    bellman_ford(loop, "A")
except ValueError as exc:
    print(f"on a negative cycle: {exc}")
    print("  ^ going round it forever drives the cost to -infinity, so")
    print("    there IS no shortest path. Saying so is the correct answer.")

## Topological sort - dependency order

*"In what order can I build these, given what depends on what?"* — package installs, build systems, task schedulers, course prerequisites, spreadsheet recalculation.

Defined only for a **DAG**: with a cycle there is no valid order, and detecting that is half the value.

**Kahn's algorithm** is BFS on in-degrees:

```
   1. count how many edges point INTO each vertex
   2. queue every vertex with in-degree 0        (nothing blocks it)
   3. take one, output it, decrement its neighbours' in-degrees
   4. any neighbour that reaches 0 joins the queue
   5. if you output fewer than V vertices -> there is a CYCLE
```

That last line is the cycle detector, free.

> The result is **not unique** — several orders are usually valid. If you need a deterministic one, use a heap instead of a queue to break ties consistently.

In [ ]:
def topological_sort(graph):
    """Kahn's algorithm. Returns an order, or raises if there is a cycle."""
    in_degree = {vertex: 0 for vertex in graph.vertices}
    for source in graph.adjacent:
        for target, _weight in graph.adjacent[source]:
            in_degree[target] += 1

    ready = deque(sorted(v for v in graph.vertices if in_degree[v] == 0))
    order = []
    while ready:
        node = ready.popleft()
        order.append(node)
        for neighbour, _weight in graph.adjacent[node]:
            in_degree[neighbour] -= 1
            if in_degree[neighbour] == 0:
                ready.append(neighbour)

    if len(order) != len(graph.vertices):
        remaining = sorted(set(graph.vertices) - set(order))
        raise ValueError(f"cycle detected among: {remaining}")
    return order


order = topological_sort(services)
print("a safe build order:", " -> ".join(order))

position = {name: i for i, name in enumerate(order)}
valid = all(position[source] < position[target] for source, target in EDGES)
print("every dependency comes before its dependents:", valid)

print("\nto deploy safely you would REVERSE it (leaves first):")
print(" ", " -> ".join(reversed(order)))

broken = Graph(directed=True)
for source, target in (("a", "b"), ("b", "c"), ("c", "a"), ("c", "d")):
    broken.add_edge(source, target)
try:
    topological_sort(broken)
except ValueError as exc:
    print(f"\n  on a cyclic graph: {exc}")
    print("  ^ exactly the CI check suggested in 10.6")

## 🔴 Cycle detection differs by graph type

This catches people, because the obvious algorithm for one is wrong for the other.

### Undirected: track the parent

Every edge `A—B` can be walked both ways, so arriving at `A` from `B` and seeing `B` again is **not** a cycle. Ignore the vertex you just came from.

### Directed: three colours

Tracking the parent is not enough — a directed graph can revisit a vertex legitimately without a cycle (a diamond). What matters is whether the vertex is **currently on the recursion stack**.

```
    WHITE  not visited yet
    GREY   visited, still being explored  <- an edge to GREY means a CYCLE
    BLACK  fully explored, all descendants done
```

An edge to a **black** vertex is fine — you have reached it another way, and it is finished. An edge to a **grey** vertex means you have looped back into the path you are still walking.

The service graph is the perfect illustration: `userdb` is reachable from `web` by three routes, and that is not a cycle.

In [ ]:
WHITE, GREY, BLACK = 0, 1, 2


def has_cycle_directed(graph):
    """Three-colour DFS. An edge to a GREY vertex closes a cycle."""
    colour = {vertex: WHITE for vertex in graph.vertices}

    def visit(node, path):
        colour[node] = GREY
        path.append(node)
        for neighbour in graph.neighbours(node):
            if colour[neighbour] == GREY:                 # back edge
                return path[path.index(neighbour):] + [neighbour]
            if colour[neighbour] == WHITE:
                found = visit(neighbour, path)
                if found:
                    return found
        colour[node] = BLACK
        path.pop()
        return None

    for vertex in sorted(graph.vertices):
        if colour[vertex] == WHITE:
            found = visit(vertex, [])
            if found:
                return found
    return None


def has_cycle_undirected(graph):
    """Track the parent: coming back the way you arrived is not a cycle."""
    visited = set()

    def visit(node, came_from):
        visited.add(node)
        for neighbour in graph.neighbours(node):
            if neighbour == came_from:      # 🔴 the edge we just used
                continue
            if neighbour in visited or visit(neighbour, node):
                return True
        return False

    return any(visit(v, None) for v in sorted(graph.vertices) if v not in visited)


print("service graph (a DAG with three routes to userdb):")
print("  cycle?", has_cycle_directed(services))
print("  ^ None. Reaching userdb three ways is NOT a cycle - which is")
print("    exactly what the three-colour method gets right and a plain")
print("    'have I seen this before?' check gets wrong.\n")

looped = Graph(directed=True)
for source, target in EDGES + [("userdb", "web")]:
    looped.add_edge(source, target)
print("after adding userdb -> web:")
print("  cycle:", " -> ".join(has_cycle_directed(looped)))

tree_like = Graph(directed=False)
for source, target in (("a", "b"), ("b", "c"), ("b", "d")):
    tree_like.add_edge(source, target)
ringed = Graph(directed=False)
for source, target in (("a", "b"), ("b", "c"), ("c", "a")):
    ringed.add_edge(source, target)

print("\nundirected:")
print("  a tree (a-b, b-c, b-d) has a cycle?", has_cycle_undirected(tree_like))
print("  a ring (a-b, b-c, c-a) has a cycle?", has_cycle_undirected(ringed))
print("\n  🔴 Without the parent check, the tree would report a cycle -")
print("     because b sees a again along the very edge it arrived on.")

## Connected components

*"How many separate islands are there, and who is on each?"* — clustering, network partitions, "number of islands" grid problems.

Run a traversal from each unvisited vertex; each run discovers exactly one component.

| | Time |
|---|---|
| BFS/DFS per component | **O(V + E)** total — every vertex and edge touched once |

🔴 In a **directed** graph, "connected" splits in two: *weakly* connected (ignoring direction) and *strongly* connected (every vertex reaches every other, respecting direction). Strongly connected components need Tarjan's or Kosaraju's algorithm — worth naming, rarely worth implementing in an interview.

> **Union-Find** solves the undirected version incrementally, as edges arrive, in nearly O(1) per operation. That is **15.15**.

In [ ]:
def connected_components(graph):
    """For an UNDIRECTED graph. O(V + E)."""
    seen = set()
    components = []
    for vertex in sorted(graph.vertices):
        if vertex in seen:
            continue
        group, frontier = [], deque([vertex])
        seen.add(vertex)
        while frontier:
            node = frontier.popleft()
            group.append(node)
            for neighbour in graph.neighbours(node):
                if neighbour not in seen:
                    seen.add(neighbour)
                    frontier.append(neighbour)
        components.append(sorted(group))
    return components


islands = Graph(directed=False)
for source, target in (("a", "b"), ("b", "c"), ("d", "e"), ("f", "f")):
    islands.add_edge(source, target)

groups = connected_components(islands)
print(f"{len(groups)} components:")
for group in groups:
    print("  ", group)


def count_islands(grid):
    """The grid version: a graph where neighbours are implicit."""
    if not grid:
        return 0
    rows, cols = len(grid), len(grid[0])
    seen = set()
    islands = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != "1" or (r, c) in seen:
                continue
            islands += 1
            frontier = deque([(r, c)])
            seen.add((r, c))
            while frontier:
                row, col = frontier.popleft()
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = row + dr, col + dc
                    if (0 <= nr < rows and 0 <= nc < cols
                            and grid[nr][nc] == "1" and (nr, nc) not in seen):
                        seen.add((nr, nc))
                        frontier.append((nr, nc))
    return islands


grid = [
    list("11000"),
    list("11000"),
    list("00100"),
    list("00011"),
]
print("\ngrid:")
for row in grid:
    print("  ", "".join(row))
print("islands:", count_islands(grid))
print("\n  🔴 'Number of islands' IS connected components. The graph is")
print("     implicit - neighbours are the four adjacent cells, and there")
print("     is no adjacency list to build.")

## Interview questions

**1. BFS vs DFS — when would you use each?**
> BFS for shortest paths and level-order; DFS for cycle detection, topological sort and path existence. BFS uses O(width) memory, DFS O(depth).

**2. Why does BFS find the shortest path but DFS does not?**
> BFS explores in order of distance, so the first time it reaches a vertex is by a shortest route. Unweighted only.

**3. Number of islands.** *(above)*
> Connected components on an implicit grid graph. O(rows × cols).

**4. Course schedule — can all courses be finished?**
> Cycle detection on a directed graph, or a topological sort that outputs fewer than V vertices.

**5. Clone a graph.**
> DFS or BFS with a `{original: copy}` map — which doubles as the visited set. Without it, a cycle loops forever.

**6. Word ladder — fewest one-letter changes from A to B.**
> BFS on an implicit graph where edges are one-letter substitutions. Build neighbours with wildcard patterns rather than comparing every pair.

**7. Implement Dijkstra.** *(above)*
> Heap-based, with lazy deletion. Say up front that it requires non-negative weights.

**8. Detect a cycle in a directed graph.** *(above)*
> Three colours. Explain why a plain visited set is wrong — a diamond revisits without a cycle.

**9. Topological sort.** *(above)*
> Kahn's algorithm, or DFS post-order reversed. The count check detects cycles for free.

**10. Adjacency list or matrix?**
> List for sparse graphs — O(V+E) space. Matrix for dense graphs or O(1) edge lookups. Real graphs are usually sparse.

**11. Find if a path exists between two nodes.**
> Either traversal — DFS is usually simpler and uses less memory if the graph is wide.

**12. Bipartite graph check / graph colouring with two colours.**
> BFS assigning alternating colours; a conflict means an odd-length cycle, so it is not bipartite.

In [ ]:
# Questions 5 and 12 - both short, both commonly asked.
def clone_graph(graph, start):
    """The {original: copy} map IS the visited set."""
    copies = {start: Graph(directed=graph.directed)}
    clone = copies[start]
    seen = {start}
    frontier = deque([start])
    while frontier:
        node = frontier.popleft()
        for neighbour, weight in graph.weighted_neighbours(node):
            clone.add_edge(node, neighbour, weight)
            if neighbour not in seen:
                seen.add(neighbour)
                frontier.append(neighbour)
    return clone


copy = clone_graph(services, "web")
print("cloned:", copy)
print("  same traversal:", bfs(copy, "web") == bfs(services, "web"))
print("  independent   :", copy.adjacent is not services.adjacent)


def is_bipartite(graph):
    """Two-colour with BFS. A conflict means an odd cycle."""
    colour = {}
    for start in sorted(graph.vertices):
        if start in colour:
            continue
        colour[start] = 0
        frontier = deque([start])
        while frontier:
            node = frontier.popleft()
            for neighbour in graph.neighbours(node):
                if neighbour not in colour:
                    colour[neighbour] = 1 - colour[node]
                    frontier.append(neighbour)
                elif colour[neighbour] == colour[node]:
                    return False, None
    return True, colour


square = Graph(directed=False)
for source, target in (("a", "b"), ("b", "c"), ("c", "d"), ("d", "a")):
    square.add_edge(source, target)

triangle = Graph(directed=False)
for source, target in (("a", "b"), ("b", "c"), ("c", "a")):
    triangle.add_edge(source, target)

for label, g in (("4-cycle (even)", square), ("3-cycle (odd)", triangle)):
    ok, colouring = is_bipartite(g)
    detail = colouring if ok else "conflict found"
    print(f"\n  {label:<16} bipartite: {str(ok):<6} {detail}")
print("\n  A graph is bipartite exactly when it has no ODD-length cycle.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Traversing without a visited set.** A cycle loops forever. Trees are safe; graphs are not.
2. 🔴 **Marking visited on dequeue rather than enqueue.** The same vertex enters the queue many times - redundant work, and it can blow up on dense graphs.
3. 🔴 **Using a plain visited set to detect cycles in a DIRECTED graph.** A diamond revisits legitimately. Use three colours.
4. 🔴 **Forgetting the parent check in an UNDIRECTED cycle check.** Every edge would look like a cycle.
5. 🔴 **Running Dijkstra with negative weights.** It finalises vertices and never revisits. Use Bellman-Ford.
6. **Using BFS for shortest paths in a WEIGHTED graph.** Fewest hops is not cheapest.
7. **Building an adjacency matrix for a sparse graph.** O(V²) memory for O(E) edges.
8. **Recursive DFS on a large graph.** Depth can be O(V); the limit is ~1000 (**15.1**).
9. **Assuming a topological order is unique.** Usually several are valid.
10. **Using `list.pop(0)` as the BFS queue.** O(n) each - use `deque` (**15.5**).

## Best Practices

- Default to an adjacency list; `defaultdict(list)` is the natural form.
- Add the visited set before writing the traversal, not after debugging it.
- Mark vertices visited as you enqueue them.
- Use BFS for unweighted shortest paths, Dijkstra when edges have costs.
- State the weight assumption whenever you propose Dijkstra.
- Prefer iterative DFS when depth could exceed the recursion limit.
- Use Kahn's algorithm when you want the order *and* cycle detection together.
- Recognise implicit graphs - grids, word ladders, state machines - and do not build an adjacency list you do not need.
- Test: a disconnected graph, a self-loop, a single vertex, and an empty graph.

## Practice Exercises

Try these before moving on.

1. Extend `bfs` to also return the level of each vertex, and confirm it matches `distances_from`.
2. 🔴 Remove the visited set from `bfs` and run it on the cyclic graph with a step limit. Then remove the limit and predict what happens before running it.
3. Implement topological sort with DFS post-order reversed, and check it agrees with Kahn's on the service graph.
4. Add edge weights to the service graph representing latency, and find the lowest-latency path from `web` to `userdb` (**10.6**).
5. Implement 'word ladder' with BFS. How do you generate neighbours without comparing every pair of words?
6. Implement Kosaraju's algorithm for strongly connected components - two DFS passes, the second on the reversed graph.
7. Make `count_islands` use DFS instead of BFS. Which uses less memory on a long thin island, and why?
8. 🔴 Build a random graph of 100,000 vertices and time BFS against recursive DFS. What happens to the recursive version, and why (**15.1**)?